# Week 22: Audit transfer before calling it a foundation model

Does source training improve target dynamics after matching labels, representation and optimizer budget?

**Prerequisites:** Week 7.4 already introduces diverse pretraining. This capstone audits mechanisms and claims rather than repeating that introduction.

Allow 120 minutes for the guided lab; estimate 3–5 hours total with reading and written analysis. This planning estimate has not been measured in a student pilot. Student edition: worked examples run immediately; set RUN_EXERCISES=True after completing the four functions. Incomplete tasks are reported, never counted as passes.

In [ ]:
from pathlib import Path
import sys, tempfile, json
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'flowmllab/transformer_course.py').exists()),None)
if ROOT is None:
    raise RuntimeError('Extract the supplied course bundle and start Jupyter in its root; see docs/TRANSFORMER_COURSE.md for Colab upload setup.')
sys.path.insert(0,str(ROOT))
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from flowmllab import transformer_course as lab
lab.seed_all(17)
cases,input_manifest=lab.load_data(ROOT)
EVIDENCE=ROOT/'results/transformer_course_v3'
# Any optional experiment must write into this disposable workspace.
workspace=tempfile.TemporaryDirectory(prefix='flowmllab-student-')
RUN_OUTPUT=Path(workspace.name)
task_status={}
print('Loaded checksummed simulated wake fields. No retained files will be rewritten.')

task_status={i:None for i in range(1,5)}


In [ ]:
RUN_EXERCISES=False

## Count every target label and fix the forecast start

Inspect every neural arm at step zero. Scratch, matched-steps and target-POD-MLP are freshly initialized when their selected step is zero; pretrained step zero means no target adaptation after source fitting. Source validation uses a 50-step rollout and target validation six-step blocks. All three retained source fits selected step 300/300, so source checkpoint selection did not alter the final source weights in these runs. Different window coverage, validation horizons and budgets prevent a single-factor causal attribution of transfer gain.

In [ ]:
for blocks in (4,8,15):
    train,val,labels=lab.transfer_indices(blocks)
    print(blocks,'blocks:',len(labels['training_frames']),'training +',len(labels['validation_frames']),'validation +',len(labels['initialization_frames']),'initialization =',labels['total_target_labels'])
record=json.loads((EVIDENCE/'transfer.json').read_text())
rows=record['rows'];display(pd.DataFrame([{'Re':r['target_re'],'labels':r['labels']['total_target_labels'],'arm':r['arm'],'seed':r['seed'],'field':r['metrics']['field_relative_l2'],'floor':r['metrics']['representation_floor'],'subspace':r['metrics']['in_subspace_relative_l2'],'steps':r.get('total_optimizer_steps'),'selected_step':(r.get('training') or {}).get('selected_step'),'boundary':(r.get('training') or {}).get('at_budget_boundary'),'no_target_adaptation':r['arm']=='pretrained' and r.get('training',{}).get('selected_step')==0,'control_untrained':r['arm'] in ('scratch','matched-steps','target-POD-MLP') and (r.get('training') or {}).get('selected_step')==0} for r in rows]))

## Recompute transfer metrics from stored predictions

Source-POD arms isolate initialization of dynamics. The target-POD MLP is a representation control, not the same architecture. Matched steps are reported alongside measured fit seconds; steps are not FLOPs. Source pretraining is reused and its amortization depends on the number of deployments.

In [ ]:
retained=np.load(EVIDENCE/'predictions.npz',allow_pickle=False)
row=next(r for r in rows if r['arm']=='pretrained' and r['target_re']==110)
pred=lab.prediction(retained,'transfer__'+row['key'])[50:]
truth=cases[110]['omega'][210:].reshape(71,-1)
score=np.linalg.norm(np.asarray(pred,dtype=np.float64)-np.asarray(truth,dtype=np.float64))/np.linalg.norm(np.asarray(truth,dtype=np.float64))
assert np.isclose(score,row['metrics']['field_relative_l2'],rtol=1e-6,atol=1e-9)
print('Recomputed W22 transfer metric:',score)
fig,axes=plt.subplots(2,3,figsize=(15,8),sharex=True)
for col,target_re in enumerate(sorted({r['target_re'] for r in rows})):
    for arm in ('scratch','pretrained','matched-steps','target-POD-MLP','DMD'):
        subset=[r for r in rows if r['arm']==arm and r['target_re']==target_re]
        for j,key in enumerate(('field_relative_l2','in_subspace_relative_l2')):
            groups={n:[r['metrics'][key] for r in subset if r['labels']['total_target_labels']==n] for n in (44,84,154)}
            means=np.array([np.mean(v) for v in groups.values()])
            low=np.array([min(v) for v in groups.values()]);high=np.array([max(v) for v in groups.values()])
            axes[j,col].errorbar(list(groups),means,yerr=np.maximum(0,np.array([means-low,high-means])),marker='o',capsize=3,label=arm)
    source_floor=next(r['metrics']['representation_floor'] for r in rows if r['target_re']==target_re and r['arm']=='pretrained')
    axes[0,col].axhline(source_floor,color='black',linestyle='--',label='source-POD floor (source-basis arms)')
    floors={n:[r['metrics']['representation_floor'] for r in rows if r['target_re']==target_re and r['arm']=='target-POD-MLP' and r['labels']['total_target_labels']==n] for n in (44,84,154)}
    axes[0,col].plot(list(floors),[np.mean(v) for v in floors.values()],':',color='black',label='target-POD floor (target-basis arm)')
    for j,title in enumerate(('field error','in-subspace error')):
        axes[j,col].set(title=f'Re={target_re}: {title}',xlabel='all target labels');axes[j,col].legend(fontsize=6)
fig.suptitle('Means and min–max seed ranges; step-zero flags remain in the table')
fig.tight_layout();plt.show()
resources=[]
for r in rows:
    tr=r.get('training') or {};pre=r.get('pretraining') or {}
    resources.append({'Re':r['target_re'],'labels':r['labels']['total_target_labels'],'arm':r['arm'],'seed':r['seed'],
        'executed_updates':r.get('total_optimizer_steps',0),'selected_step':tr.get('selected_step'),
        'parameters':tr.get('parameters'),'seconds_adaptation':tr.get('seconds'),
        'seconds_pretraining':pre.get('seconds'),'source_fields_accessed':r['arm']=='pretrained',
        'source_basis_accessed':r['arm']!='target-POD-MLP'})
display(pd.DataFrame(resources))
print('Equal update caps do not match FLOPs, wall time, or source information. Missing resource fields are unavailable, never zero cost.')

## Task 1: Audit information budgets

Return the cardinality of the union of all target frames used before scoring.

Interface contract: record has training_frames, validation_frames and initialization_frames, each an iterable of integer frame IDs. Return an integer cardinality of their union, counting every overlap once.

In [ ]:
def label_count(record):
    """record has training_frames, validation_frames and initialization_frames, each an iterable of integer frame IDs. Return an integer cardinality of their union, counting every overlap once."""
    raise NotImplementedError

In [ ]:
if RUN_EXERCISES:
    try:
        for blocks in (4,8,15):assert label_count(lab.transfer_indices(blocks)[2])==blocks*10+4
        assert label_count({'training_frames':[1,2],'validation_frames':[2,3],'initialization_frames':[3,4]})==4
        task_status[1]=True
    except NotImplementedError:
        task_status[1]=None
    except Exception as exc:
        task_status[1]=False
        print("Task 1 check failed:",type(exc).__name__,str(exc))
else:
    task_status[1]=None
print("Task 1:","PASS" if task_status[1] is True else "NOT SUBMITTED" if task_status[1] is None else "FAILED")

## Task 2: Match compute accounting

Count source pretraining and target adaptation steps for each learned arm, without charging source cost to a source-free arm.

Interface contract: adaptation is a dictionary with executed_steps (nonnegative integer). pretraining is None or a dictionary with executed_steps. Return their sum when pretraining is supplied, otherwise adaptation executed_steps.

In [ ]:
def total_steps(adapt,pretrain=None):
    """adaptation is a dictionary with executed_steps (nonnegative integer). pretraining is None or a dictionary with executed_steps. Return their sum when pretraining is supplied, otherwise adaptation executed_steps."""
    raise NotImplementedError

In [ ]:
if RUN_EXERCISES:
    try:
        for adapt,source in ((3,7),(23,0),(0,11)):
            assert total_steps({'executed_steps':adapt},{'executed_steps':source})==adapt+source
        assert total_steps({'executed_steps':13})==13
        task_status[2]=True
    except NotImplementedError:
        task_status[2]=None
    except Exception as exc:
        task_status[2]=False
        print("Task 2 check failed:",type(exc).__name__,str(exc))
else:
    task_status[2]=None
print("Task 2:","PASS" if task_status[2] is True else "NOT SUBMITTED" if task_status[2] is None else "FAILED")

## Task 3: Detect a representation-limited result

Flag cases where the squared representation error explains more than 95% of squared field error. Do not subtract unsquared norms.

Interface contract: metrics is a dictionary with representation_floor (floor) and field_relative_l2 (field), both nonnegative scalar relative L2 errors. Return a bool for floor**2/max(field**2,1e-20)>0.95. At zero/zero return False; use a strict threshold.

In [ ]:
def representation_limited(metrics):
    """metrics is a dictionary with representation_floor (floor) and field_relative_l2 (field), both nonnegative scalar relative L2 errors. Return a bool for floor**2/max(field**2,1e-20)>0.95. At zero/zero return False; use a strict threshold."""
    raise NotImplementedError

In [ ]:
if RUN_EXERCISES:
    try:
        assert representation_limited({'representation_floor':.2559,'field_relative_l2':.256})
        assert not representation_limited({'representation_floor':.1,'field_relative_l2':.5})
        assert not representation_limited({'representation_floor':0.,'field_relative_l2':0.})
        assert not representation_limited({'representation_floor':.20,'field_relative_l2':.21})
        assert representation_limited({'representation_floor':.21,'field_relative_l2':.21})
        task_status[3]=True
    except NotImplementedError:
        task_status[3]=None
    except Exception as exc:
        task_status[3]=False
        print("Task 3 check failed:",type(exc).__name__,str(exc))
else:
    task_status[3]=None
print("Task 3:","PASS" if task_status[3] is True else "NOT SUBMITTED" if task_status[3] is None else "FAILED")

## Task 4: Build a defensible transfer comparison

For records with key (target, labels, seed), pair pretrained and matched-steps arms. Reject missing or duplicate arms and nonpositive control error. Return signed relative gain, selected-step flags and budget-boundary flags. Treat step-zero scratch/matched results as untrained controls. Apply the audit to a normalized copy of the retained table; explain why this does not isolate representation or validation-horizon effects.

Interface contract: Each input dictionary has target, labels (integer total target labels), seed, arm, error (in-subspace relative L2), selected_step, ceiling. Pair pretrained with matched-steps per (target,labels,seed), rejecting duplicates/missing arms or nonpositive control error with ValueError. Return a list sorted by group key of dictionaries: key (tuple), gain (1-pretrained_error/control_error), control_untrained (control selected step==0), pretrained_selected_step, control_selected_step, pretrained_boundary and control_boundary (each arm selected step==its own ceiling). Keep negative gains.

In [ ]:
def audit_transfer_comparisons(records):
    """Each input dictionary has target, labels (integer total target labels), seed, arm, error (in-subspace relative L2), selected_step, ceiling. Pair pretrained with matched-steps per (target,labels,seed), rejecting duplicates/missing arms or nonpositive control error with ValueError. Return a list sorted by group key of dictionaries: key (tuple), gain (1-pretrained_error/control_error), control_untrained (control selected step==0), pretrained_selected_step, control_selected_step, pretrained_boundary and control_boundary (each arm selected step==its own ceiling). Keep negative gains."""
    raise NotImplementedError

In [ ]:
if RUN_EXERCISES:
    try:
        records=[]
        for target,pre,control,selected in ((100,.02,.01,0),(110,.01,.02,8)):
            for arm,error in [('pretrained',pre),('matched-steps',control)]:
                records.append({'target':target,'labels':44,'seed':17,'arm':arm,'error':error,'selected_step':selected,'ceiling':8})
        r=audit_transfer_comparisons(records)
        assert [v['gain'] for v in r]==[-1.,.5]
        assert r[0]['control_untrained'] and not r[1]['control_untrained']
        assert r[1]['control_boundary'] and not r[0]['control_boundary']
        for invalid in (records[:-1],records+[records[0]]):
            try:
                audit_transfer_comparisons(invalid)
                raise AssertionError('invalid pairing accepted')
            except ValueError:pass
        records=[{'target':105,'labels':84,'seed':29,'arm':'pretrained','error':.03,'selected_step':7,'ceiling':7},
                 {'target':105,'labels':84,'seed':29,'arm':'matched-steps','error':.04,'selected_step':2,'ceiling':11}]
        r=audit_transfer_comparisons(records)[0]
        assert r['pretrained_selected_step']==7 and r['control_selected_step']==2
        assert r['pretrained_boundary'] is True and r['control_boundary'] is False
        assert r['control_untrained'] is False and np.isclose(r['gain'],.25)
        records[0]['selected_step']=0;records[1]['selected_step']=11
        r=audit_transfer_comparisons(records)[0]
        assert r['pretrained_boundary'] is False and r['control_boundary'] is True
        assert r['control_untrained'] is False
        for invalid in (0.,-.1):
            records[1]['error']=invalid
            try:
                audit_transfer_comparisons(records)
                raise AssertionError('nonpositive control error accepted')
            except ValueError:pass
        task_status[4]=True
    except NotImplementedError:
        task_status[4]=None
    except Exception as exc:
        task_status[4]=False
        print("Task 4 check failed:",type(exc).__name__,str(exc))
else:
    task_status[4]=None
print("Task 4:","PASS" if task_status[4] is True else "NOT SUBMITTED" if task_status[4] is None else "FAILED")

## Assessment: 100 points

Coding Tasks 1–4: [5, 5, 5, 25] points; derivation 25, interpretation 25, reproducibility 10. Automated checks assess only coding behavior.

Derive: derive each target-label budget including initialization.

Worked interpretation: A pretrained step-zero selection uses source dynamics without target adaptation, although target validation selected it. Compare paired seeds and report negative gains with target and source floors.

Reject and explain: “Matched update caps prove equal compute and information.”

An accurate numerical observation without its split, representation, or causal limitation earns at most half of interpretation points. Instructor marking and a timed student pilot are still required; automated checks establish interface/behavior only.

Interpretation (25 points): 10 for source-information accounting, including source-POD access by scratch and matched-steps controls even without source neural pretraining; 10 for separating optimizer updates from FLOPs, measured adaptation/pretraining time and amortization across targets; 5 for paired-seed signed gains, negative transfer and the retained-case scope.

## Interpretation and submission

Submit the four completed functions, the requested plots/tables, and a 300-word claim ledger. Distinguish observations, fitted quantities, independent evaluation and unsupported extrapolation. Retain failed seeds. Use the lecture source for the rubric and assumptions.

In [ ]:
print("Coding tasks passed:",sum(v is True for v in task_status.values()),"/",len(task_status))
workspace.cleanup()